In [127]:
import random
import time
import math

import pandas as pd
from pandas import DataFrame
import numpy as np
from scipy.spatial import cKDTree


from preferences import prefs
from user import user_prefs

# startPoint = {"lat": 30.317504,"lon": 59.927085}
# endPoint = {"lat": 30.327108, "lon": 59.935408}

POPULATION_SIZE = 50
GENERATIONS = 50

MIN_ROUTE_POINTS = 3
MAX_ROUTE_POINTS = 50

KILLOMETER_RADIUS = 1
ADD_POINT_KILLOMETER_RADIUS = 0.3

MUTATION_RATE = 0.5

PRINT = True

INTEREST_K = 10
DISTANCE_K = 0.1

# random.seed(42)
# np.random.seed(42)

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [128]:
df = pd.read_csv('data/places.csv')

coords = df[["lat", "lon"]].to_numpy()
tree = cKDTree(coords)

**Алгоритмы генетики**

In [129]:
from haversine import haversine

# Оптимизация конкретного маршрута
def optimize_route(route_ids, df, startPoint):
    points = (
        df[df["id"].isin(route_ids)]
        [["id", "lat", "lon"]]
        .copy()
    )

    remaining = points.to_dict("records")

    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    optimized_route = []

    while remaining:
        nearest = min(
            remaining,
            key=lambda p: haversine(
                (current["lat"], current["lon"]),
                (p["lat"], p["lon"])
            )
        )

        optimized_route.append(nearest["id"])

        current = nearest

        remaining.remove(nearest)

    return optimized_route
    
df_filtered=0
# Генерация случайной особи
def create_population(df: DataFrame, startPoint: dict[str, float], endPoint: dict[str, float], num: int):
    global df_filtered
    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    radius_deg = KILLOMETER_RADIUS / 111.0

    start = np.array([
        startPoint["lat"],
        startPoint["lon"]
    ])

    finish = np.array([
        endPoint["lat"],
        endPoint["lon"]
    ])

    # Длина маршрута в километрах
    mean_lat = (start[0] + finish[0]) / 2

    lat_km = (finish[0] - start[0]) * 111
    lon_km = (
        (finish[1] - start[1])
        * 111
        * np.cos(np.radians(mean_lat))
    )

    route_length_km = np.hypot(lat_km, lon_km)

    # Точка каждые ~500 метров
    samples_count = max(
        2,
        int(route_length_km / 0.5)
    )

    route_points = np.linspace(
        start,
        finish,
        samples_count
    )

    indices = set()

    for point in route_points:
        nearby = tree.query_ball_point(
            point,
            r=radius_deg
        )

        indices.update(nearby)

    df_filtered = df.iloc[list(indices)]

    df_indexes = df_filtered["id"].tolist()

    if len(df_indexes) < route_size:
        route_size = len(df_indexes)

    population = []

    for _ in range(num):
        route_ids = random.sample(
            df_indexes,
            route_size
        )

        new_route = optimize_route(
            route_ids,
            df,
            startPoint
        )

        population.append({
            "route": new_route,
            "fitness": None
        })

    return population

# Длина маршрута
def route_distance(route: list, startPoint: dict[str, float], endPoint: dict[str, float]):
    
    route_df = pd.DataFrame()

    for j in route:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    remaining = route_df.to_dict(orient='index')
    current = {
        "lat": startPoint["lat"],
        "lon": startPoint["lon"]
    }

    distance = haversine((current["lat"], current["lon"]),
                         (remaining[0]["lat"], remaining[0]["lon"]))

    current = remaining[0]

    for point_id in remaining.keys():
        if point_id == 0:
            continue
        distance += haversine((current["lat"], current["lon"]),
                              (remaining[point_id]["lat"], remaining[point_id]["lon"]))
        current = remaining[point_id]
    distance = haversine((current["lat"], current["lon"]),
                         (endPoint["lat"], endPoint["lon"]))
    return distance


# Интересность конкретной точки
def point_interest_score(point: dict, unique_user_prefs: dict[str, int]):
    point_tags = next(iter(point.values()))
    score = 0
    for category, tags in prefs.items():
        weight = unique_user_prefs[category]
        for tag in tags:
            if tag in point_tags and point_tags[tag]:
                score += weight
    return score

# Фитнес-функция
def fitness_function(route: list, unique_user_prefs: dict[str, int], startPoint: dict[str, float], endPoint: dict[str, float]):

    total_interest = 0

    for place_id in route:
        point = df.iloc[(df["id"] == place_id)].to_dict(orient='index')
        total_interest += point_interest_score(point, unique_user_prefs)

    distance = route_distance(route, startPoint, endPoint)

    fitness = total_interest * INTEREST_K - distance * DISTANCE_K

    return fitness

# Проведение оценки популяции (просчёт фитнес-функций каждой особи)
def evaluate_population(population: dict, unique_user_prefs: dict[str, int], startPoint: dict[str, float], endPoint: dict[str, float]):
    for individual in population:
        individual["fitness"] = fitness_function(
            individual["route"],
            unique_user_prefs,
            startPoint,
            endPoint
        )


# Мутация
def mutate(individual, unique_user_prefs: dict[str, int], startPoint: dict[str, float]):
    if random.random() > MUTATION_RATE:
        return individual
    
    route = individual["route"][:]
    route_df = pd.DataFrame()

    for j in route:
        route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

    mutation_type = random.choice([
        "remove_minimum_score",
        "remove_most_distant",
        "add_nearby_point"
    ])

    if len(route) >= MAX_ROUTE_POINTS-1:
        mutation_type = random.choice([
            "remove_minimum_score",
            "remove_most_distant"
        ])

    if mutation_type == "remove_minimum_score":
        if len(route) > MIN_ROUTE_POINTS:
            min_score_point = route[0]
            
            min_score = point_interest_score({0:df.iloc[0].to_dict()}, unique_user_prefs)
            for place_id in route:
                new_min_score = point_interest_score({0:df.iloc[(df["id"] == place_id)].to_dict()}, unique_user_prefs)
                if min_score > new_min_score:
                    min_score = new_min_score
                    min_score_point = place_id
            
            route.remove(min_score_point)

    elif mutation_type == "remove_most_distant":
        if len(route) > MIN_ROUTE_POINTS:
            sum_lat = 0
            sum_lon = 0
            for index, row in route_df.iterrows():
                sum_lat += row["lat"]
                sum_lon += row["lon"]
            mean_lat = sum_lat / len(route)
            mean_lon = sum_lon / len(route)
            
            max_distance_point = route[0]
            
            max_distance = haversine((mean_lat, mean_lon), (route_df["lat"].iloc[0], route_df["lon"].iloc[0]))
            
            for place_id in route:
                place = route_df.loc[route_df["id"] == place_id].iloc[0]

                new_max_distance = haversine((mean_lat, mean_lon),(place["lat"], place["lon"]))
                
                if max_distance < new_max_distance:
                    max_distance = new_max_distance
                    max_distance_point = place_id
            
            route.remove(max_distance_point)
    
    elif mutation_type == "add_nearby_point":
        if len(route) < MAX_ROUTE_POINTS:
            source_id = random.choice(route)

            source_point = route_df.loc[
                route_df["id"] == source_id
            ].iloc[0]

            lat = source_point["lat"]
            lon = source_point["lon"]

            radius_deg = ADD_POINT_KILLOMETER_RADIUS / 111.0

            indices = tree.query_ball_point(
                [lat, lon],
                r=radius_deg
            )

            neighbors_df = df.iloc[indices]

            # id найденных точек
            neighbors_id = neighbors_df["id"].tolist()

            # исключаем уже существующие в маршруте
            candidates = [
                place_id
                for place_id in neighbors_id
                if place_id not in route
            ]

            if candidates:
                best_point = max(candidates,
                                key=lambda place_id:
                                    point_interest_score(
                                        {0: df.loc[df["id"] == place_id].iloc[0].to_dict()},
                                        unique_user_prefs
                                    )
                                )
                route.append(best_point)
    
    route = optimize_route(route, df, startPoint)
    
    return {"route": route, "fitness": None}

# Создание нового поколения
def create_next_generation(population: dict, unique_user_prefs: dict[str, int], startPoint: dict[str, float]):
    new_population = []

    population.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    # elite = population[:math.ceil(POPULATION_SIZE/10)]
    elite = population[:10]

    new_population.extend(elite)

    while len(new_population) < POPULATION_SIZE:
        child = mutate(random.choice(elite), unique_user_prefs, startPoint)
        new_population.append(child)

    return new_population

In [130]:
import folium

def hex_to_rgb(hex_str):
    """Преобразует HEX в кортеж RGB (0-255)"""
    hex_str = hex_str.lstrip('#')
    return tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb):
    """Преобразует RGB в HEX строку"""
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]), int(rgb[1]), int(rgb[2]))

def generate_gradient(start_hex, end_hex, steps):
    """Генерирует список цветов для градиента"""
    start_rgb = hex_to_rgb(start_hex)
    end_rgb = hex_to_rgb(end_hex)
    
    gradient_colors = []
    for i in range(steps):
        t = i / max(1, steps - 1)
        r = start_rgb[0] + (end_rgb[0] - start_rgb[0]) * t
        g = start_rgb[1] + (end_rgb[1] - start_rgb[1]) * t
        b = start_rgb[2] + (end_rgb[2] - start_rgb[2]) * t
        gradient_colors.append(rgb_to_hex((r, g, b)))
        
    return gradient_colors

def visual(all_routes: bool, population: dict):

    path_colors = generate_gradient("#FF0000","#0000FF", len(population))

    route_df = df[(df["id"].isin(population[0]["route"]))]
    m = folium.Map(
        location=[
            route_df["lat"].mean(),
            route_df["lon"].mean()
        ],
        zoom_start=14
    )
    if all_routes:
        for i in range(len(population)):
            individual = population[i]
            
            route_df = pd.DataFrame()

            for j in individual["route"]:
                route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

            # Точки маршрута
            folium.Marker(
                [startPoint["lat"], startPoint["lon"]]
            ).add_to(m)
            folium.Marker(
                [endPoint["lat"], endPoint["lon"]]
            ).add_to(m)

            route_line = [[startPoint["lat"], startPoint["lon"]]]
            route_line.extend(route_df[["lat", "lon"]].values.tolist())
            route_line.append([endPoint["lat"], endPoint["lon"]])

            # Линия маршрута
            folium.PolyLine(
                route_line,
                weight=4,
                color=path_colors[i]
            ).add_to(m)

        m.save(r"visualisations\route.html")
    else:
        individual = population[0]

        route_df = pd.DataFrame()

        for j in individual["route"]:
            route_df = pd.concat([route_df, df[(df["id"] == j)]], ignore_index=True)

        # Точки маршрута
        folium.Marker(
            [startPoint["lat"], startPoint["lon"]]
        ).add_to(m)
        folium.Marker(
            [endPoint["lat"], endPoint["lon"]]
        ).add_to(m)

        route_line = [[startPoint["lat"], startPoint["lon"]]]
        route_line.extend(route_df[["lat", "lon"]].values.tolist())
        route_line.append([endPoint["lat"], endPoint["lon"]])

        # Линия маршрута
        folium.PolyLine(
            route_line,
            weight=4,
            color=path_colors[0]
        ).add_to(m)

        m.save(r"visualisations\route.html")

def visual_df_filtered(startPoint):

    m = folium.Map(
        location=[
            startPoint["lat"],
            startPoint["lon"]
        ],
        zoom_start=15
    )
    
    # Точки маршрута
    for i in df_filtered.values:
        folium.Marker(
            [i[1], i[2]]
        ).add_to(m)

    m.save(r"visualisations\filtered_points.html")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path


def render_generation_fast(
    generation_idx,
    population,
    startPoint,
    endPoint,
    all_routes=True
):
    output_dir = Path("visualisations/jpg")
    output_dir.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 10))

    # фоновые точки (POI)
    plt.scatter(
        df["lon"],
        df["lat"],
        s=2,
        alpha=0.3
    )

    # старт / финиш
    plt.scatter(
        startPoint["lon"],
        startPoint["lat"],
        c="green",
        s=80,
        label="Start"
    )

    plt.scatter(
        endPoint["lon"],
        endPoint["lat"],
        c="red",
        s=80,
        label="End"
    )

    # градиент цветов маршрутов
    colors = generate_gradient("#FF0000", "#0000FF", len(population))

    def draw_route(route, color):
        route_df = df[df["id"].isin(route["route"])]

        xs = [startPoint["lon"]] + route_df["lon"].tolist() + [endPoint["lon"]]
        ys = [startPoint["lat"]] + route_df["lat"].tolist() + [endPoint["lat"]]

        plt.plot(xs, ys, color=color, linewidth=2, alpha=0.9)

    if all_routes:
        for i, individual in enumerate(population):
            draw_route(individual, colors[i])
    else:
        draw_route(population[0], colors[0])

    plt.title(f"Generation {generation_idx}")
    plt.legend()

    plt.xlim(df["lon"].min(), df["lon"].max())
    plt.ylim(df["lat"].min(), df["lat"].max())

    plt.tight_layout()

    path = output_dir / f"generation_{generation_idx}.jpg"
    plt.savefig(path, dpi=200)
    plt.close()

    return path

**Запуск генетики**

In [ ]:
startPoint = {"lat": 59.927085,"lon": 30.317504}
# endPoint = {"lat": 59.935408, "lon": 30.327108}
endPoint = {"lat": 59.948936, "lon": 30.374199}

population = create_population(df, startPoint, endPoint, POPULATION_SIZE)

visual_df_filtered(startPoint)

unique_user_prefs = user_prefs


print("Начало работы")
start_time = time.time()

for generation in range(GENERATIONS):
    evaluate_population(population, unique_user_prefs, startPoint, endPoint)

    best = max(
        population,
        key=lambda x: x["fitness"]
    )

    if PRINT:
        print(
            f"Поколение {generation} | "
            f"Лучшая фитнес-функция: {best['fitness']:.10f} | "
            f"Длина маршрута: {len(best['route'])}"
    )
    population = create_next_generation(population, unique_user_prefs, startPoint)
    visual(True, population)
    render_generation_fast(
        generation,
        population,
        startPoint,
        endPoint,
        all_routes=True
    )

work_time = time.time() - start_time
print("Время:", work_time)

visual(False, population)
render_generation_fast(
        -1,
        population,
        startPoint,
        endPoint,
        all_routes=False
    )

Начало работы
Поколение 0 | Лучшая фитнес-функция: 39.6996904188 | Длина маршрута: 46
Поколение 1 | Лучшая фитнес-функция: 39.6996904188 | Длина маршрута: 46
Поколение 2 | Лучшая фитнес-функция: 49.6996904188 | Длина маршрута: 47
Поколение 3 | Лучшая фитнес-функция: 49.9432426696 | Длина маршрута: 48
Поколение 4 | Лучшая фитнес-функция: 59.6996904188 | Длина маршрута: 48
Поколение 5 | Лучшая фитнес-функция: 59.6996904188 | Длина маршрута: 48
Поколение 6 | Лучшая фитнес-функция: 69.6996904188 | Длина маршрута: 49
Поколение 7 | Лучшая фитнес-функция: 69.6996904188 | Длина маршрута: 49
Поколение 8 | Лучшая фитнес-функция: 79.6996904188 | Длина маршрута: 49
Поколение 9 | Лучшая фитнес-функция: 79.6996904188 | Длина маршрута: 49
Поколение 10 | Лучшая фитнес-функция: 79.6996904188 | Длина маршрута: 49
Поколение 11 | Лучшая фитнес-функция: 79.6996904188 | Длина маршрута: 49
Поколение 12 | Лучшая фитнес-функция: 79.6996904188 | Длина маршрута: 49
Поколение 13 | Лучшая фитнес-функция: 79.699690